# Impact of Dried Blood Spot Diameter and Transit Time on Analyte Concentrations and Cut-Off Classification in Routine Newborn Screening

## Modelling impact of DBS diameter and transit time on result classification

This Jupyter notebook describes the analysis of newborn screening result to assess the impact of transit time and DBS diameter on interpretation against UK newborn screning cut-off values, part of the following manuscript:

{Citation placeholder - Once accepted for publication, a citation will appear here}

Please cite this paper if you re-use any of the code in this analysis.

### Library imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import math
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from statsmodels.stats.proportion import proportion_confint

In [2]:
sns.set_style("whitegrid")

# Aim

Calculate how often DBS diameter and transit time may have changed classification

# Initial

Import initial results for samples received between April 2015 and October 2025

In [3]:
tsh = pd.read_csv('data/chtpopulationsd_0415-1025.csv', parse_dates=["Date Rec'd", "Sample Date"], dayfirst=True, low_memory=False)
irt = pd.read_csv('data/cfpopulationsd_0415-1025.csv', parse_dates=["Date Rec'd", "Sample Date"], dayfirst=True, low_memory=False)
imd = pd.read_csv('data/imdpopulationsd_0415-1025.csv', parse_dates=["Date Rec'd", "Sample Date"], dayfirst=True, low_memory=False)

In [4]:
def report_counts(tshdf, irtdf, imddf):
    print("TSH: " + str(len(tshdf)))
    print("IRT: " + str(len(irtdf)))
    print("IMD: " + str(len(imddf)))

In [5]:
report_counts(tsh,irt,imd)

TSH: 277012
IRT: 265295
IMD: 274039


Remove commas from numbers for IMDs

In [6]:
imd_cols = ['C10', 'C8', 'Phe', 'Tyr', 'C5', 'C5DC', 'Met', 'Xleu']

In [7]:
for col in imd_cols:
    imd[col] = (
        imd[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

In [8]:
report_counts(tsh,irt,imd)

TSH: 277012
IRT: 265295
IMD: 274039


Dataframes include greater than and less than values. Replace with numeric placehold values then convert column to numeric

In [9]:
def handle_lt_gt(df,analyte):
    # Ensure string dtype
    df[analyte] = df[analyte].astype(str)
    
    # Handle < values
    mask_lt = df[analyte].str.startswith("<", na=False)
    df.loc[mask_lt, analyte] = (
    df.loc[mask_lt, analyte]
       .str.replace("<", "", regex=False)
       .astype(float) * 1
    )

    # Handle > values
    mask_gt = df[analyte].str.startswith(">", na=False)
    df.loc[mask_gt, analyte] = (
    df.loc[mask_gt, analyte]
       .str.replace(">", "", regex=False)
       .astype(float) * 1
    )
    
    df[analyte] = pd.to_numeric(df[analyte], errors="coerce")

In [10]:
handle_lt_gt(tsh,"TSH")
handle_lt_gt(irt,"IRT")

for col in imd_cols:
    handle_lt_gt(imd,col)
    
report_counts(tsh,irt,imd)

TSH: 277012
IRT: 265295
IMD: 274039


Convert dates to datetimes (and rename column name for IMD)

In [11]:
tsh["Analysis date"] = pd.to_datetime(tsh["Date Rec'd"], errors="coerce")
irt["Analysis date"] = pd.to_datetime(irt["Date Rec'd"], errors="coerce")
imd["Analysis date"] = pd.to_datetime(imd["Date Rec'd"], errors="coerce")

report_counts(tsh,irt,imd)

TSH: 277012
IRT: 265295
IMD: 274039


Drop any rows with missing sample date

In [12]:
tsh = tsh[~tsh["Sample Date"].isna()]
irt = irt[~irt["Sample Date"].isna()]
imd = imd[~imd["Sample Date"].isna()]

report_counts(tshdf=tsh,irtdf=irt,imddf=imd)

TSH: 275921
IRT: 265295
IMD: 272972


Calculate transit time

In [13]:
tsh["Days to receive"] = ((tsh["Analysis date"] - tsh["Sample Date"]).dt.days).astype("Int64") 
irt["Days to receive"] = ((irt["Analysis date"] - irt["Sample Date"]).dt.days).astype("Int64") 
imd["Days to receive"] = ((imd["Analysis date"] - imd["Sample Date"]).dt.days).astype("Int64") 

report_counts(tshdf=tsh,irtdf=irt,imddf=imd)

TSH: 275921
IRT: 265295
IMD: 272972


### Load DBS quality metrics

Load DBS metrics, obtained using methods described in the following paper.

    Flynn N, Moat SJ, Hogg SL. A computer vision approach to the assessment of dried blood spot size and quality in newborn screening. Clin Chim Acta. 2023;547:117418.

Data is stored by Panthera instrument and month, so we load each each month separately before combining into a single dataframe. 

Note this is for all images obtained from the Panthera. We will merge with just the samples we need later.

In [14]:
year = (2015,2016,2017,2018,2019,2020,2021,2022,2023)
month = (1,2,3,4,5,6,7,8,9,10,11,12)
dataframes = []

for y in year:
    for m in month:
        try:
            filepath = 'data/dbs_metrics/' + str(y) + '-' + str(m) + '.csv'
            df_month = pd.read_csv(filepath,parse_dates=['month','date_time'])
            dataframes.append(df_month)
        except:
            print('No data for year ' + str(y) + ' and month ' + str(m))

No data for year 2015 and month 1
No data for year 2023 and month 11
No data for year 2023 and month 12


In [15]:
year = (2023,2024,2025)
month = (1,2,3,4,5,6,7,8,9,10,11,12)

for y in year:
    for m in month:
        try:
            filepath = 'data/dbs_metrics/' + str(y) + '-' + str(m) + '_new.csv'
            df_month = pd.read_csv(filepath,parse_dates=['month','date_time'])
            dataframes.append(df_month)
        except:
            print('No data for year ' + str(y) + ' and month ' + str(m))

No data for year 2023 and month 1
No data for year 2023 and month 2
No data for year 2023 and month 3
No data for year 2023 and month 4
No data for year 2023 and month 5
No data for year 2023 and month 6
No data for year 2023 and month 7
No data for year 2023 and month 8
No data for year 2023 and month 9
No data for year 2025 and month 11
No data for year 2025 and month 12


In [16]:
dbs_metrics = pd.concat(dataframes,ignore_index=True)

In [17]:
len(dbs_metrics)

423390

Add column to define whether DBS diameter falls within acceptable range (8-14mm)

In [18]:
dbs_metrics['diam_8_to_14'] = dbs_metrics['equiv_diam_mm'].between(8, 14)

Add columns for total and running total of punches, and the starting number of punches - these will be used to work out which image corresponds to which punch (assuming IMD first, TSH second and IRT third)

First convert number of punches to numeric datatype

In [19]:
dbs_metrics['number_punches'] = pd.to_numeric(
    dbs_metrics['number_punches'], errors='coerce'
)

In [20]:
g = dbs_metrics.groupby('Episode_pseudo')

totals = g['number_punches'].transform('sum')
running = g['number_punches'].cumsum()

dbs_metrics['total_punches'] = totals
dbs_metrics['running_punches'] = running
dbs_metrics['start_punch'] = running - dbs_metrics['number_punches']

IMD is punched first, so is include if we have start without any punches

In [21]:
dbs_metrics['IMDp'] = False
dbs_metrics.loc[(dbs_metrics['start_punch'] == 0)
    & (dbs_metrics['number_punches'] >= 1),'IMDp'] = True

TSH is punched second, so is included if we have more than 2 punches, start without any punches and punch more than one punch, or begin after punching 1 spot and punch any number of punches.

Note this won't include repeat samples collected due to a previous borderline result or  prem repeat

In [22]:
dbs_metrics['TSHp'] = False
dbs_metrics.loc[(dbs_metrics['start_punch'] == 0)
    & (dbs_metrics['number_punches'] >= 2),'TSHp'] = True

dbs_metrics.loc[(dbs_metrics['start_punch'] == 1)
    & (dbs_metrics['number_punches'] >= 1),'TSHp'] = True

IRT is punched third, so is included if we have more than 3 punches in total, we start without any punches and punch 3 or more punches, after 1 punch and punch 2 or more, or after 2 punches and punch 1

In [23]:
dbs_metrics['IRTp'] = False
dbs_metrics.loc[(dbs_metrics['start_punch'] == 0)
    & (dbs_metrics['number_punches'] >= 3),'IRTp'] = True
                 
dbs_metrics.loc[(dbs_metrics['start_punch'] == 1)
    & (dbs_metrics['number_punches'] >= 2),'IRTp'] = True
                 
dbs_metrics.loc[(dbs_metrics['start_punch'] == 2)
    & (dbs_metrics['number_punches'] >= 1),'IRTp'] = True

Exclude incorrect blood application samples

In [24]:
dbs_metrics = dbs_metrics[dbs_metrics['pred_multi'] == 'controls']

In [25]:
dbs_metrics['equiv_diam_mm'].describe()

count    392881.000000
mean         10.286866
std           1.209270
min           2.796439
25%           9.523859
50%          10.330938
75%          11.072829
max          19.204941
Name: equiv_diam_mm, dtype: float64

Use the median DBS diameter in the entire dataset as the reference

In [26]:
reference_mm = dbs_metrics['equiv_diam_mm'].median()
reference_mm

10.330938304229155

In [27]:
imd_dbsmetrics = dbs_metrics[dbs_metrics['IMDp'] == True].copy()
tsh_dbsmetrics = dbs_metrics[dbs_metrics['TSHp'] == True].copy()
irt_dbsmetrics = dbs_metrics[dbs_metrics['IRTp'] == True].copy()

report_counts(tsh_dbsmetrics,irt_dbsmetrics,imd_dbsmetrics)

TSH: 261558
IRT: 257616
IMD: 265785


Check we don't have any duplicate episode numbers

In [28]:
tsh_dbsmetrics['Episode_pseudo'].value_counts().max()

1

In [29]:
irt_dbsmetrics['Episode_pseudo'].value_counts().max()

1

In [30]:
imd_dbsmetrics['Episode_pseudo'].value_counts().max()

1

Merge with initial results

In [31]:
tsh = tsh.merge(tsh_dbsmetrics,left_on='Episode_pseudo',right_on='Episode_pseudo')
irt = irt.merge(irt_dbsmetrics,left_on='Episode_pseudo',right_on='Episode_pseudo')
imd = imd.merge(imd_dbsmetrics,left_on='Episode_pseudo',right_on='Episode_pseudo')

report_counts(tsh,irt,imd)

TSH: 256466
IRT: 246449
IMD: 256554


Drop any TSH results before May 2018, as results only recorded to integer values

In [32]:
tsh = tsh[tsh["Analysis date"] > "2018-05-07"]

report_counts(tsh,irt,imd)

TSH: 180702
IRT: 246449
IMD: 256554


Calculate DBS diameter correction factors

In [33]:
IMD_cols = ['C10', 'C8', 'Phe', 'Tyr', 'C5', 'C5DC', 'Met','Xleu']

tsh_dbs_adj = 0.0303  ## from paired DBS experiments
irt_dbs_adj = 0.0305  ## from paired DBS experiments
imd_dbs_adj_dict  = {
    'C10':-0.0037, 'C8': 0.0104, 'Phe': 0.0148, 'Tyr': 0.0069,
    'C5':0.0025 , 'C5DC': 0.0055 , 'Met': 0.0142, 'Xleu': 0.0246
}

adj_factor_low = 0.015
adj_factor_high = 0.03

## TSH 
tsh['diff_from_ref'] = reference_mm-tsh['equiv_diam_mm']
tsh['tsh_dbs_cf_ols'] = 1/(1-(tsh_dbs_adj*tsh['diff_from_ref'])) # from paired DBS experiments
tsh['tsh_dbs_cf_low'] = 1/(1-(adj_factor_low*tsh['diff_from_ref'])) #sensitivity analysis (low)
tsh['tsh_dbs_cf_high'] = 1/(1-(adj_factor_high*tsh['diff_from_ref'])) # sensitivity analysis (high)

## IRT
irt['diff_from_ref'] = reference_mm-irt['equiv_diam_mm']
irt['irt_dbs_cf_ols'] = 1/(1-(irt_dbs_adj*irt['diff_from_ref']))  # from paired DBS experiments
irt['irt_dbs_cf_low'] = 1/(1-(adj_factor_low*irt['diff_from_ref'])) #sensitivity analysis (low)
irt['irt_dbs_cf_high'] = 1/(1-(adj_factor_high*irt['diff_from_ref']))  # sensitivity analysis (high)

## IMD

imd['diff_from_ref'] = reference_mm-imd['equiv_diam_mm']

for analyte, adj in imd_dbs_adj_dict.items():
    col_name = f'{analyte}_dbs_cf_ols'
    imd[col_name] = 1 / (1 - (adj * (imd['diff_from_ref']))) ### from initial results OLS

    imd[f'{analyte}_dbs_cf_low'] = (
        1 / (1 - adj_factor_low * imd['diff_from_ref']) ## sensitivity analysis (low)
    )
    imd[f'{analyte}_dbs_cf_high'] = (
        1 / (1 - adj_factor_high * imd['diff_from_ref']) ## sensitivity analysis (high)
    )

Calculate transit time correction factors to a reference transit time of 2 days (the median in the dataset)

In [34]:
reference_trans = 2

## TSH 
tsh_transit_adj = -0.0253
tsh['tsh_transit_cf'] = 1/(1-(tsh_transit_adj*(reference_trans - tsh['Days to receive'])))

## IRT
irt_transit_adj = -0.0085
irt['irt_transit_cf'] = 1/(1-(irt_transit_adj*(reference_trans - irt['Days to receive'])))


## IMD analytes
imd_transit_adj_dict = {
    'C10': -0.0015, 'C8': -0.0041, 'Phe': -0.0043, 'Tyr': -0.0092,
    'C5': -0.0048, 'C5DC': -0.0148, 'Met': -0.0379, 'Xleu': 0.0047
}

for analyte, adj in imd_transit_adj_dict.items():
    col_name = f'{analyte}_transit_cf'
    imd[col_name] = 1 / (1 - (adj * (reference_trans - imd['Days to receive'])))

Calculate combined correction factors (including contributions from both DBS diameter and transit time)

In [35]:
## TSH
tsh['tsh_cf_combined_ols'] = tsh['tsh_dbs_cf_ols']*tsh['tsh_transit_cf']
tsh['tsh_cf_combined_low'] = tsh['tsh_dbs_cf_low']*tsh['tsh_transit_cf'] # Sensitivity analysis (low)
tsh['tsh_cf_combined_high'] = tsh['tsh_dbs_cf_high']*tsh['tsh_transit_cf'] # Sensitivity analysis (high)

## TSH
irt['irt_cf_combined_ols'] = irt['irt_dbs_cf_ols']*irt['irt_transit_cf']
irt['irt_cf_combined_low'] = irt['irt_dbs_cf_low']*irt['irt_transit_cf'] # Sensitivity analysis (low)
irt['irt_cf_combined_high'] = irt['irt_dbs_cf_high']*irt['irt_transit_cf'] # Sensitivity analysis (high)

## IMD

IMD_cols = ['C10', 'C8', 'Phe', 'Tyr', 'C5', 'C5DC', 'Met','Xleu']

for analyte in IMD_cols:
    transit_col = f'{analyte}_transit_cf'
    imd[f'{analyte}_cf_combined_ols'] = imd[f'{analyte}_dbs_cf_ols'] * imd[transit_col]
    
    imd[f'{analyte}_cf_combined_low'] = imd[f'{analyte}_dbs_cf_low'] * imd[transit_col]  # Sensitivity analysis (low)
    imd[f'{analyte}_cf_combined_high'] = imd[f'{analyte}_dbs_cf_high'] * imd[transit_col] # Sensitivity analysis (high)

Now calculate concentrations

In [36]:
## TSH
tsh['tsh_dbs_corrected_ols'] = tsh['TSH'] * tsh['tsh_dbs_cf_ols']
tsh['tsh_dbs_corrected_low'] = tsh['TSH'] * tsh['tsh_dbs_cf_low']
tsh['tsh_dbs_corrected_high'] = tsh['TSH'] * tsh['tsh_dbs_cf_high']

tsh['tsh_transit_corrected'] = tsh['TSH'] * tsh['tsh_transit_cf']

tsh['tsh_combined_ols'] = tsh['TSH'] * tsh['tsh_cf_combined_ols']
tsh['tsh_combined_low'] = tsh['TSH'] * tsh['tsh_cf_combined_low']
tsh['tsh_combined_high'] = tsh['TSH'] * tsh['tsh_cf_combined_high']

## IRT
irt['irt_dbs_corrected_ols'] = irt['IRT'] * irt['irt_dbs_cf_ols']
irt['irt_dbs_corrected_low'] = irt['IRT'] * irt['irt_dbs_cf_low']
irt['irt_dbs_corrected_high'] = irt['IRT'] * irt['irt_dbs_cf_high']

irt['irt_transit_corrected'] = irt['IRT'] * irt['irt_transit_cf']

irt['irt_combined_ols'] = irt['IRT'] * irt['irt_cf_combined_ols']
irt['irt_combined_low'] = irt['IRT'] * irt['irt_cf_combined_low']
irt['irt_combined_high'] = irt['IRT'] * irt['irt_cf_combined_high']

## IMD analytes
IMD_cols = ['C10', 'C8', 'Phe', 'Tyr', 'C5', 'C5DC', 'Met', 'Xleu']

for analyte in IMD_cols:
    # DBS corrected
    imd[f'{analyte}_dbs_corrected_ols'] = imd[analyte] * imd[f'{analyte}_dbs_cf_ols']
    imd[f'{analyte}_dbs_corrected_low'] = imd[analyte] * imd[f'{analyte}_dbs_cf_low']
    imd[f'{analyte}_dbs_corrected_high'] = imd[analyte] * imd[f'{analyte}_dbs_cf_high']
    
    # Transit corrected
    imd[f'{analyte}_transit_corrected'] = imd[analyte] * imd[f'{analyte}_transit_cf']
    
    # Combined corrected
    imd[f'{analyte}_combined_ols'] = imd[analyte] * imd[f'{analyte}_cf_combined_ols']
    imd[f'{analyte}_combined_low'] = imd[analyte] * imd[f'{analyte}_cf_combined_low']
    imd[f'{analyte}_combined_high'] = imd[analyte] * imd[f'{analyte}_cf_combined_high']


Check the transit calculation looks sensible

In [37]:
tsh[['TSH','Days to receive','tsh_transit_corrected']]

,TSH,Days to receive,tsh_transit_corrected
75764,0.8,4,0.842637
75765,1.2,1,1.170389
75766,1.4,4,1.474616
75767,1.2,4,1.263956
75768,1.6,4,1.685275
...,...,...,...
256461,3.4,1,3.316103
256462,1.9,1,1.853116
256463,0.9,1,0.877792
256464,6.2,1,6.047011


Now classify results against cut-off values

In [39]:
def classify_cutoff(series, cutoff, unit):
    return series.apply(
        lambda x: (
            f'< {cutoff} {unit}'
            if x < cutoff else
            f'>= {cutoff} {unit}'
        )
    )

Classification for TSH

In [41]:
TSH_COV = 8
TSH_UNIT = 'mU/L'

# Raw classification
tsh['classification'] = classify_cutoff(tsh['TSH'], TSH_COV, TSH_UNIT)

# DBS-corrected classifications
for level in ['ols','low','high']:
    col = f'tsh_dbs_corrected_{level}'
    tsh[f'classification_adjusted_{level}'] = classify_cutoff(tsh[col], TSH_COV, TSH_UNIT)
    tsh[f'{level}_agree'] = tsh['classification'] == tsh[f'classification_adjusted_{level}']

# Transit-corrected classification
tsh['classification_transit'] = classify_cutoff(tsh['tsh_transit_corrected'], TSH_COV, TSH_UNIT)
tsh['transit_agree'] = tsh['classification'] == tsh['classification_transit']

# Combined-corrected classifications
for level in ['ols','low','high']:
    col = f'tsh_combined_{level}'
    tsh[f'classification_combined_{level}'] = classify_cutoff(tsh[col], TSH_COV, TSH_UNIT)
    tsh[f'{level}_combined_agree'] = tsh['classification'] == tsh[f'classification_combined_{level}']


IRT

In [42]:
IRT_COV = 56
IRT_UNIT = 'ng/mL'

# Raw classification
irt['classification'] = classify_cutoff(irt['IRT'], IRT_COV, IRT_UNIT)

# DBS-corrected classifications
for level in ['ols','low','high']:
    col = f'irt_dbs_corrected_{level}'
    irt[f'classification_adjusted_{level}'] = classify_cutoff(irt[col], IRT_COV, IRT_UNIT)
    irt[f'{level}_agree'] = irt['classification'] == irt[f'classification_adjusted_{level}']

# Transit-corrected classification
irt['classification_transit'] = classify_cutoff(irt['irt_transit_corrected'], IRT_COV, IRT_UNIT)
irt['transit_agree'] = irt['classification'] == irt['classification_transit']

# Combined-corrected classifications
for level in ['ols','low','high']:
    col = f'irt_combined_{level}'
    irt[f'classification_combined_{level}'] = classify_cutoff(irt[col], IRT_COV, IRT_UNIT)
    irt[f'{level}_combined_agree'] = irt['classification'] == irt[f'classification_combined_{level}']


IMD

In [43]:
IMD_cols = ['C10','C8','Phe','Tyr','C5','C5DC','Met','Xleu']
IMD_UNIT = 'µmol/L'
IMD_CUTOFFS = {
    'C10': 0.5,
    'C8': 0.5,
    'Phe': 240,
    'Tyr': 240,
    'C5': 2.0,
    'C5DC': 0.70,
    'Met': 50,
    'Xleu': 600
}

for col in IMD_cols:
    COV = IMD_CUTOFFS[col]

    # Raw classification
    imd[f'{col}_classification'] = classify_cutoff(imd[col], COV, IMD_UNIT)

    # DBS-corrected classifications
    for level in ['ols','low','high']:
        corrected_col = f'{col}_dbs_corrected_{level}'
        class_col = f'{col}_classification_adjusted_{level}'
        imd[class_col] = classify_cutoff(imd[corrected_col], COV, IMD_UNIT)
        imd[f'{col}_{level}_agree'] = imd[f'{col}_classification'] == imd[class_col]

    # Transit-corrected classification
    transit_col = f'{col}_transit_corrected'
    class_col = f'{col}_classification_transit'
    imd[class_col] = classify_cutoff(imd[transit_col], COV, IMD_UNIT)
    imd[f'{col}_transit_agree'] = imd[f'{col}_classification'] == imd[class_col]

    # Combined-corrected classifications
    for level in ['ols','low','high']:
        combined_col = f'{col}_combined_{level}'
        class_col = f'{col}_classification_combined_{level}'
        imd[class_col] = classify_cutoff(imd[combined_col], COV, IMD_UNIT)
        imd[f'{col}_{level}_combined_agree'] = imd[f'{col}_classification'] == imd[class_col]

## Number of changed classifications

Calculate birth rate in UK using published data

In [44]:
#source https://www.ons.gov.uk/peoplepopulationandcommunity/birthsdeathsandmarriages/livebirths/bulletins/birthsummarytablesenglandandwales/2024refreshedpopulations
birthrate_ew = 594677

# National Records of Scotland
birthrate_s = 45793

# Source - 
birthrate_ni = 19416 

birthrate = birthrate_ew + birthrate_s + birthrate_ni

birthrate

659886

In [45]:
def generate_agreement_summary(tsh, irt, imd):
    """
    Generate a summary of agreement differences for TSH, IRT, and IMD analytes,
    including DBS-corrected, transit-corrected, and combined-corrected adjustments.
    """
    
    rows = []

    # ----------------------------------------
    # TSH
    # ----------------------------------------
    total = len(tsh)

    # DBS-corrected
    for level in ['ols','low','high']:
        agree_col = f'{level}_agree'
        if agree_col not in tsh.columns:
            continue
        num_diff = (~tsh[agree_col]).sum()
        percent_different = (num_diff / total) * 100
        ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
        rows.append({
            'analyte': 'TSH',
            'adjustment': f'DBS_{level}',
            'total': total,
            'num_differences': num_diff,
            'percent_different': percent_different,
            'ci_lower_percent': ci_lower*100,
            'ci_upper_percent': ci_upper*100,
            'annual_incidence': (num_diff/total)*birthrate,
            'ci_lower_incidence': ci_lower*birthrate,
            'ci_upper_incidence': ci_upper*birthrate
        })

    # Transit-corrected
    if 'classification_transit' in tsh.columns:
        num_diff = (~(tsh['classification'] == tsh['classification_transit'])).sum()
        percent_different = (num_diff / total) * 100
        ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
        rows.append({
            'analyte': 'TSH',
            'adjustment': 'Transit',
            'total': total,
            'num_differences': num_diff,
            'percent_different': percent_different,
            'ci_lower_percent': ci_lower*100,
            'ci_upper_percent': ci_upper*100,
            'annual_incidence': (num_diff/total)*birthrate,
            'ci_lower_incidence': ci_lower*birthrate,
            'ci_upper_incidence': ci_upper*birthrate
        })

    # Combined-corrected
    for level in ['ols','low','high']:
        agree_col = f'{level}_combined_agree'
        if agree_col not in tsh.columns:
            continue
        num_diff = (~tsh[agree_col]).sum()
        percent_different = (num_diff / total) * 100
        ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
        rows.append({
            'analyte': 'TSH',
            'adjustment': f'Combined_{level}',
            'total': total,
            'num_differences': num_diff,
            'percent_different': percent_different,
            'ci_lower_percent': ci_lower*100,
            'ci_upper_percent': ci_upper*100,
            'annual_incidence': (num_diff/total)*birthrate,
            'ci_lower_incidence': ci_lower*birthrate,
            'ci_upper_incidence': ci_upper*birthrate
        })

    # ----------------------------------------
    # IRT
    # ----------------------------------------
    total = len(irt)

    # DBS-corrected
    for level in ['ols','low','high']:
        agree_col = f'{level}_agree'
        if agree_col not in irt.columns:
            continue
        num_diff = (~irt[agree_col]).sum()
        percent_different = (num_diff / total) * 100
        ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
        rows.append({
            'analyte': 'IRT',
            'adjustment': f'DBS_{level}',
            'total': total,
            'num_differences': num_diff,
            'percent_different': percent_different,
            'ci_lower_percent': ci_lower*100,
            'ci_upper_percent': ci_upper*100,
            'annual_incidence': (num_diff/total)*birthrate,
            'ci_lower_incidence': ci_lower*birthrate,
            'ci_upper_incidence': ci_upper*birthrate
        })

    # Transit-corrected
    if 'classification_transit' in irt.columns:
        num_diff = (~(irt['classification'] == irt['classification_transit'])).sum()
        percent_different = (num_diff / total) * 100
        ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
        rows.append({
            'analyte': 'IRT',
            'adjustment': 'Transit',
            'total': total,
            'num_differences': num_diff,
            'percent_different': percent_different,
            'ci_lower_percent': ci_lower*100,
            'ci_upper_percent': ci_upper*100,
            'annual_incidence': (num_diff/total)*birthrate,
            'ci_lower_incidence': ci_lower*birthrate,
            'ci_upper_incidence': ci_upper*birthrate
        })

    # Combined-corrected
    for level in ['ols','low','high']:
        agree_col = f'{level}_combined_agree'
        if agree_col not in irt.columns:
            continue
        num_diff = (~irt[agree_col]).sum()
        percent_different = (num_diff / total) * 100
        ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
        rows.append({
            'analyte': 'IRT',
            'adjustment': f'Combined_{level}',
            'total': total,
            'num_differences': num_diff,
            'percent_different': percent_different,
            'ci_lower_percent': ci_lower*100,
            'ci_upper_percent': ci_upper*100,
            'annual_incidence': (num_diff/total)*birthrate,
            'ci_lower_incidence': ci_lower*birthrate,
            'ci_upper_incidence': ci_upper*birthrate
        })

    # ----------------------------------------
    # IMD analytes
    # ----------------------------------------
    total = len(imd)

    for col in imd_cols:
        # DBS-corrected
        for level in ['ols','low','high']:
            agree_col = f'{col}_{level}_agree'
            if agree_col not in imd.columns:
                continue
            num_diff = (~imd[agree_col]).sum()
            percent_different = (num_diff / total) * 100
            ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
            rows.append({
                'analyte': col,
                'adjustment': f'DBS_{level}',
                'total': total,
                'num_differences': num_diff,
                'percent_different': percent_different,
                'ci_lower_percent': ci_lower*100,
                'ci_upper_percent': ci_upper*100,
                'annual_incidence': (num_diff/total)*birthrate,
                'ci_lower_incidence': ci_lower*birthrate,
                'ci_upper_incidence': ci_upper*birthrate
            })

        # Transit-corrected
        transit_agree = f'{col}_transit_agree'
        if transit_agree in imd.columns:
            num_diff = (~imd[transit_agree]).sum()
            percent_different = (num_diff / total) * 100
            ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
            rows.append({
                'analyte': col,
                'adjustment': 'Transit',
                'total': total,
                'num_differences': num_diff,
                'percent_different': percent_different,
                'ci_lower_percent': ci_lower*100,
                'ci_upper_percent': ci_upper*100,
                'annual_incidence': (num_diff/total)*birthrate,
                'ci_lower_incidence': ci_lower*birthrate,
                'ci_upper_incidence': ci_upper*birthrate
            })

        # Combined-corrected
        for level in ['ols','low','high']:
            agree_col = f'{col}_{level}_combined_agree'
            if agree_col not in imd.columns:
                continue
            num_diff = (~imd[agree_col]).sum()
            percent_different = (num_diff / total) * 100
            ci_lower, ci_upper = proportion_confint(count=num_diff, nobs=total, alpha=0.05, method='beta')
            rows.append({
                'analyte': col,
                'adjustment': f'Combined_{level}',
                'total': total,
                'num_differences': num_diff,
                'percent_different': percent_different,
                'ci_lower_percent': ci_lower*100,
                'ci_upper_percent': ci_upper*100,
                'annual_incidence': (num_diff/total)*birthrate,
                'ci_lower_incidence': ci_lower*birthrate,
                'ci_upper_incidence': ci_upper*birthrate
            })

    # Create summary DataFrame
    summary = pd.DataFrame(rows)
    return summary


In [46]:
summary_all = generate_agreement_summary(tsh, irt, imd)

In [47]:
summary_all

,analyte,adjustment,total,num_differences,percent_different,ci_lower_percent,ci_upper_percent,annual_incidence,ci_lower_incidence,ci_upper_incidence
0,TSH,DBS_ols,180702,48,0.026563,0.019586,0.035217,175.285985,129.246200,232.393727
1,TSH,DBS_low,180702,30,0.016602,0.011202,0.023699,109.553740,73.917240,156.389226
2,TSH,DBS_high,180702,48,0.026563,0.019586,0.035217,175.285985,129.246200,232.393727
3,TSH,Transit,180702,34,0.018816,0.013031,0.026292,124.160906,85.987286,173.495934
4,TSH,Combined_ols,180702,50,0.027670,0.020538,0.036478,182.589567,135.525955,240.710978
...,...,...,...,...,...,...,...,...,...,...
65,Xleu,DBS_high,256554,1,0.000390,0.000010,0.002172,2.572113,0.065120,14.330771
66,Xleu,Transit,256554,0,0.000000,0.000000,0.001438,0.000000,0.000000,9.488148
67,Xleu,Combined_ols,256554,1,0.000390,0.000010,0.002172,2.572113,0.065120,14.330771
68,Xleu,Combined_low,256554,1,0.000390,0.000010,0.002172,2.572113,0.065120,14.330771


## Subgroup analyses

In [48]:
## Unacceptable DBS
summary_unacc_dbs_diam = generate_agreement_summary(tsh = tsh[tsh['diam_8_to_14'] == False],
                                                    irt = irt[irt['diam_8_to_14'] == False],
                                                    imd = imd[imd['diam_8_to_14'] == False])

## Acceptable DBS
summary_acc_dbs_diam = generate_agreement_summary(tsh = tsh[tsh['diam_8_to_14'] == True],
                                                    irt = irt[irt['diam_8_to_14'] == True],
                                                    imd = imd[imd['diam_8_to_14'] == True])

## Acceptable DBS
summary_transit_lt7 = generate_agreement_summary(tsh = tsh[tsh['Days to receive'] < 7],
                                                 irt = irt[irt['Days to receive'] < 7],
                                                imd = imd[imd['Days to receive'] < 7])

## Acceptable DBS
summary_transit_mt7 = generate_agreement_summary(tsh = tsh[tsh['Days to receive'] >= 7],
                                                 irt = irt[irt['Days to receive'] >= 7],
                                                imd = imd[imd['Days to receive'] >= 7])

In [49]:
summary_transit_lt7[summary_transit_lt7['analyte'] == 'TSH']

,analyte,adjustment,total,num_differences,percent_different,ci_lower_percent,ci_upper_percent,annual_incidence,ci_lower_incidence,ci_upper_incidence
0,TSH,DBS_ols,179387,48,0.026758,0.019730,0.035475,176.570922,130.193672,234.097219
1,TSH,DBS_low,179387,30,0.016724,0.011284,0.023873,110.356826,74.459105,157.535599
2,TSH,DBS_high,179387,48,0.026758,0.019730,0.035475,176.570922,130.193672,234.097219
3,TSH,Transit,179387,31,0.017281,0.011742,0.024528,114.035387,77.483443,161.858263
4,TSH,Combined_ols,179387,49,0.027315,0.020209,0.036111,180.249483,133.353927,238.288894
5,TSH,Combined_low,179387,39,0.021741,0.015460,0.029719,143.463874,102.019759,196.112014
6,TSH,Combined_high,179387,49,0.027315,0.020209,0.036111,180.249483,133.353927,238.288894


In [50]:
summary_transit_lt7[summary_transit_lt7['analyte'] == 'Met']

,analyte,adjustment,total,num_differences,percent_different,ci_lower_percent,ci_upper_percent,annual_incidence,ci_lower_incidence,ci_upper_incidence
56,Met,DBS_ols,254177,5,0.001967,0.000639,0.004591,12.980836,4.214862,30.292544
57,Met,DBS_low,254177,6,0.002361,0.000866,0.005138,15.577003,5.716517,33.904108
58,Met,DBS_high,254177,9,0.003541,0.001619,0.006722,23.365505,10.684278,44.354302
59,Met,Transit,254177,10,0.003934,0.001887,0.007235,25.961672,12.449734,47.743652
60,Met,Combined_ols,254177,11,0.004328,0.002160,0.007743,28.557840,14.256097,51.096991
61,Met,Combined_low,254177,11,0.004328,0.002160,0.007743,28.557840,14.256097,51.096991
62,Met,Combined_high,254177,13,0.005115,0.002723,0.008746,33.750174,17.970726,57.712778


In [51]:
# Dictionary of subgroup summaries
subgroups = {
    'dbs_unacc': summary_unacc_dbs_diam,
    'dbs_acc': summary_acc_dbs_diam,
    'transit_lt7': summary_transit_lt7,
    'transit_mt7': summary_transit_mt7
}

summary = summary_all.copy()

for key, df in subgroups.items():
    summary = summary.merge(
        df,
        on=['analyte','adjustment'],
        how='left',
        suffixes=('','_' + key)
    )

In [52]:
summary['risk_ratio_dbs_diam'] = summary['percent_different_dbs_unacc']/summary['percent_different_dbs_acc']
summary['risk_ratio_transit'] = summary['percent_different_transit_mt7']/summary['percent_different_transit_lt7']

Computer confidence intervals

In [53]:
from statsmodels.stats.contingency_tables import Table2x2

def compute_rr_statsmodels(row, num_diff_col1, total_col1, num_diff_col2, total_col2):
    """
    Computes risk ratio and 95% CI for a 2x2 table
    """
    a = row[num_diff_col1]
    b = row[total_col1]
    c = row[num_diff_col2]
    d = row[total_col2]

    # Skip rows where any group has zero total
    if b == 0 or d == 0:
        return pd.Series([np.nan, np.nan, np.nan])
    
    # Construct 2x2 table
    table = np.array([[a, b - a],
                      [c, d - c]])
    
    # Table2x2 object
    tb = Table2x2(table)
    
    # RR and 95% CI
    RR = tb.riskratio
    CI_lower, CI_upper = tb.riskratio_confint()
    
    return pd.Series([RR, CI_lower, CI_upper])

DBS diameter risk ratio confidence intervals

In [54]:
summary[['RR_dbs', 'RR_dbs_CI_lower', 'RR_dbs_CI_upper']] = summary.apply(
    compute_rr_statsmodels,
    axis=1,
    num_diff_col1='num_differences_dbs_unacc',
    total_col1='total_dbs_unacc',
    num_diff_col2='num_differences_dbs_acc',
    total_col2='total_dbs_acc'
)

Risk ratio for transit time

In [55]:
summary[['RR_transit', 'RR_transit_CI_lower', 'RR_transit_CI_upper']] = summary.apply(
    compute_rr_statsmodels,
    axis=1,
    num_diff_col1='num_differences_transit_mt7',
    total_col1='total_transit_mt7',
    num_diff_col2='num_differences_transit_lt7',
    total_col2='total_transit_lt7'
)

In [56]:
summary

,analyte,adjustment,total,num_differences,percent_different,ci_lower_percent,ci_upper_percent,annual_incidence,ci_lower_incidence,ci_upper_incidence,...,ci_lower_incidence_transit_mt7,ci_upper_incidence_transit_mt7,risk_ratio_dbs_diam,risk_ratio_transit,RR_dbs,RR_dbs_CI_lower,RR_dbs_CI_upper,RR_transit,RR_transit_CI_lower,RR_transit_CI_upper
0,TSH,DBS_ols,180702,48,0.026563,0.019586,0.035217,175.285985,129.246200,232.393727,...,0.000000,1848.539007,9.108352,0.000000,9.108352,2.834191,29.271871,1.420460,0.087624,23.026812
1,TSH,DBS_low,180702,30,0.016602,0.011202,0.023699,109.553740,73.917240,156.389226,...,0.000000,1848.539007,9.758949,0.000000,9.758949,2.327199,40.923476,2.272735,0.138997,37.161318
2,TSH,DBS_high,180702,48,0.026563,0.019586,0.035217,175.285985,129.246200,232.393727,...,0.000000,1848.539007,9.108352,0.000000,9.108352,2.834191,29.271871,1.420460,0.087624,23.026812
3,TSH,Transit,180702,34,0.018816,0.013031,0.026292,124.160906,85.987286,173.495934,...,310.621818,4389.905861,13.221802,13.201545,13.221802,4.047242,43.193874,13.201545,4.041033,43.127780
4,TSH,Combined_ols,180702,50,0.027670,0.020538,0.036478,182.589567,135.525955,240.710978,...,12.704720,2791.075920,8.720763,2.783999,8.720763,2.717815,27.982661,2.783999,0.384723,20.146042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Xleu,DBS_high,256554,1,0.000390,0.000010,0.002172,2.572113,0.065120,14.330771,...,0.000000,1023.286508,0.000000,0.000000,50.924033,1.708826,1517.566163,53.454679,1.793766,1592.963136
66,Xleu,Transit,256554,0,0.000000,0.000000,0.001438,0.000000,0.000000,9.488148,...,0.000000,1023.286508,NaN,NaN,101.848266,2.021328,5131.810122,106.909569,2.121797,5386.780993
67,Xleu,Combined_ols,256554,1,0.000390,0.000010,0.002172,2.572113,0.065120,14.330771,...,0.000000,1023.286508,0.000000,0.000000,50.924033,1.708826,1517.566163,53.454679,1.793766,1592.963136
68,Xleu,Combined_low,256554,1,0.000390,0.000010,0.002172,2.572113,0.065120,14.330771,...,0.000000,1023.286508,0.000000,0.000000,50.924033,1.708826,1517.566163,53.454679,1.793766,1592.963136


# Summary tables for manuscript

In [57]:
def format_percent(x):
    if 0 < x < 0.001:
        return " (<0.001%)"
    else:
        return f" ({x:.3f}%)"

In [58]:
table_summary = pd.DataFrame()
table_summary['adjustment'] = summary['adjustment']
table_summary['Analyte'] = summary['analyte']
table_summary['n'] = summary['total']
table_summary['Number changed classifications (%)'] = (
    summary['num_differences'].astype(int).astype(str)
    + summary['percent_different'].map(format_percent)
    )

table_summary['Estimated UK annual incidence (95% CI)'] = (
    summary['annual_incidence'].astype(int).astype(str) +
    + summary['ci_lower_incidence'].map(lambda x: f" ({x:.0f} - ")
    + summary['ci_upper_incidence'].map(lambda x: f"{x:.0f})")
    )

table_summary['Number acceptable DBS size (8–14mm) (%)'] = (
    summary['total_dbs_acc'].astype(int).astype(str)
    + (100* summary['total_dbs_acc'] / summary['total'])
        .map(lambda x: f" ({x:.1f}%)")
)

table_summary['Number changed classification (8-14mm subgroup) (%)'] = (
    summary['num_differences_dbs_acc'].astype(int).astype(str)
    + summary['percent_different_dbs_acc'].map(format_percent)
    )

table_summary['Number changed classification (< 8 mm or > 14 mm subgroup) (%)'] = (
    summary['num_differences_dbs_unacc'].astype(int).astype(str)
    + summary['percent_different_dbs_unacc'].map(format_percent)
    )

mask = (
    (summary['num_differences_dbs_acc'].astype(float) != 0)
    & (summary['num_differences_dbs_unacc'].astype(float) != 0)
)

table_summary['Risk ratio DBS diameter (95% CI)'] = ""

table_summary.loc[mask, 'Risk ratio DBS diameter (95% CI)'] = (
    summary.loc[mask, 'risk_ratio_dbs_diam'].astype(float).round(1).astype(str)
    + summary.loc[mask, 'RR_dbs_CI_lower'].map(lambda x: f" ({x:.1f} - ")
    + summary.loc[mask, 'RR_dbs_CI_upper'].map(lambda x: f"{x:.1f})")
)

table_summary['Number transit time < 7 days (%)'] = (
    summary['total_transit_lt7'].astype(int).astype(str)
    + (100*summary['total_transit_lt7'] / summary['total'])
        .map(lambda x: f" ({x:.1f}%)")
)

table_summary['Number changed classification (< 7 day transit time) (%)'] = (
    summary['num_differences_transit_lt7'].astype(int).astype(str)
    + summary['percent_different_transit_lt7'].map(format_percent)
    )

table_summary['Number changed classification (>= 7 day transit time) (%)'] = (
    summary['num_differences_transit_mt7'].astype(int).astype(str)
    + summary['percent_different_transit_mt7'].map(format_percent)
    )

table_summary['Risk ratio transit time (95% CI)'] = ""

mask = (
    (summary['num_differences_transit_lt7'].astype(float) != 0)
    & (summary['num_differences_transit_mt7'].astype(float) != 0)
)

table_summary.loc[mask, 'Risk ratio transit time (95% CI)'] = (
    summary.loc[mask, 'risk_ratio_transit'].astype(float).round(1).astype(str)
    + summary.loc[mask, 'RR_transit_CI_lower'].map(lambda x: f" ({x:.1f} - ")
    + summary.loc[mask, 'RR_transit_CI_upper'].map(lambda x: f"{x:.1f})")
)

table_summary

,adjustment,Analyte,n,Number changed classifications (%),Estimated UK annual incidence (95% CI),Number acceptable DBS size (8–14mm) (%),Number changed classification (8-14mm subgroup) (%),Number changed classification (< 8 mm or > 14 mm subgroup) (%),Risk ratio DBS diameter (95% CI),Number transit time < 7 days (%),Number changed classification (< 7 day transit time) (%),Number changed classification (>= 7 day transit time) (%),Risk ratio transit time (95% CI)
0,DBS_ols,TSH,180702,48 (0.027%),175 (129 - 232),179389 (99.3%),45 (0.025%),3 (0.228%),9.1 (2.8 - 29.3),179387 (99.3%),48 (0.027%),0 (0.000%),
1,DBS_low,TSH,180702,30 (0.017%),109 (74 - 156),179389 (99.3%),28 (0.016%),2 (0.152%),9.8 (2.3 - 40.9),179387 (99.3%),30 (0.017%),0 (0.000%),
2,DBS_high,TSH,180702,48 (0.027%),175 (129 - 232),179389 (99.3%),45 (0.025%),3 (0.228%),9.1 (2.8 - 29.3),179387 (99.3%),48 (0.027%),0 (0.000%),
3,Transit,TSH,180702,34 (0.019%),124 (86 - 173),179389 (99.3%),31 (0.017%),3 (0.228%),13.2 (4.0 - 43.2),179387 (99.3%),31 (0.017%),3 (0.228%),13.2 (4.0 - 43.1)
4,Combined_ols,TSH,180702,50 (0.028%),182 (136 - 241),179389 (99.3%),47 (0.026%),3 (0.228%),8.7 (2.7 - 28.0),179387 (99.3%),49 (0.027%),1 (0.076%),2.8 (0.4 - 20.1)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,DBS_high,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
66,Transit,Xleu,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
67,Combined_ols,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
68,Combined_low,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),


Drop secondary analytes and order rows

In [59]:
table_summary['adjustment'].unique()

array(['DBS_ols', 'DBS_low', 'DBS_high', 'Transit', 'Combined_ols',
       'Combined_low', 'Combined_high'], dtype=object)

In [60]:
analyte_order = ['TSH','IRT','Phe','Met','Xleu','C5','C5DC','C8']
adjustment_order = ['Combined_ols',
       'Combined_low', 'Combined_high','DBS_ols', 'DBS_low', 'DBS_high', 'Transit']

table_summary['Analyte'] = pd.Categorical(
    table_summary['Analyte'],
    categories=analyte_order,
    ordered=True
)

# Set categorical order
table_summary['adjustment'] = pd.Categorical(
    table_summary['adjustment'],
    categories=adjustment_order,
    ordered=True
)

table_summary = table_summary.sort_values(['adjustment','Analyte'])

## Table 3

In [61]:
adjustments_mid = ['Combined_ols']

table3 = table_summary[
    (table_summary['adjustment'].isin(adjustments_mid))
    & (table_summary['Analyte'].isin(analyte_order))
]

table3

,adjustment,Analyte,n,Number changed classifications (%),Estimated UK annual incidence (95% CI),Number acceptable DBS size (8–14mm) (%),Number changed classification (8-14mm subgroup) (%),Number changed classification (< 8 mm or > 14 mm subgroup) (%),Risk ratio DBS diameter (95% CI),Number transit time < 7 days (%),Number changed classification (< 7 day transit time) (%),Number changed classification (>= 7 day transit time) (%),Risk ratio transit time (95% CI)
4,Combined_ols,TSH,180702,50 (0.028%),182 (136 - 241),179389 (99.3%),47 (0.026%),3 (0.228%),8.7 (2.7 - 28.0),179387 (99.3%),49 (0.027%),1 (0.076%),2.8 (0.4 - 20.1)
11,Combined_ols,IRT,246449,153 (0.062%),409 (347 - 480),242822 (98.5%),145 (0.060%),8 (0.221%),3.7 (1.8 - 7.5),244239 (99.1%),152 (0.062%),1 (0.045%),0.7 (0.1 - 5.2)
32,Combined_ols,Phe,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
60,Combined_ols,Met,256554,16 (0.006%),41 (24 - 67),254060 (99.0%),14 (0.006%),2 (0.080%),14.6 (3.3 - 64.0),254177 (99.1%),11 (0.004%),5 (0.210%),48.6 (16.9 - 139.8)
67,Combined_ols,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
46,Combined_ols,C5,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
53,Combined_ols,C5DC,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
25,Combined_ols,C8,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),


Calculate a total incidence by summing point estimates and lower and upper confidence intervals

In [62]:
total_incidence = 182 + 409 + 41 + 2 + 2
total_lowerCI = 136 + 347 + 24
total_upperCI = 241 + 480 + 9 + 67 + 14 + 14 + 9 + 9

print(str(total_incidence) + " (" + str(total_lowerCI) + " - " + str(total_upperCI) + ")")

636 (507 - 843)


In [63]:
table3.to_csv('table3.csv', index=False)

## Supplemental table (transit and DBS effects in isolation)

In [64]:
adjustments_sep = ['DBS_ols', 'Transit']

table_sep = table_summary[
    (table_summary['adjustment'].isin(adjustments_sep))
    & (table_summary['Analyte'].isin(analyte_order))
]

table_sep

,adjustment,Analyte,n,Number changed classifications (%),Estimated UK annual incidence (95% CI),Number acceptable DBS size (8–14mm) (%),Number changed classification (8-14mm subgroup) (%),Number changed classification (< 8 mm or > 14 mm subgroup) (%),Risk ratio DBS diameter (95% CI),Number transit time < 7 days (%),Number changed classification (< 7 day transit time) (%),Number changed classification (>= 7 day transit time) (%),Risk ratio transit time (95% CI)
0,DBS_ols,TSH,180702,48 (0.027%),175 (129 - 232),179389 (99.3%),45 (0.025%),3 (0.228%),9.1 (2.8 - 29.3),179387 (99.3%),48 (0.027%),0 (0.000%),
7,DBS_ols,IRT,246449,158 (0.064%),423 (360 - 494),242822 (98.5%),150 (0.062%),8 (0.221%),3.6 (1.8 - 7.3),244239 (99.1%),158 (0.065%),0 (0.000%),
28,DBS_ols,Phe,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
56,DBS_ols,Met,256554,5 (0.002%),12 (4 - 30),254060 (99.0%),5 (0.002%),0 (0.000%),,254177 (99.1%),5 (0.002%),0 (0.000%),
63,DBS_ols,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
42,DBS_ols,C5,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
49,DBS_ols,C5DC,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
21,DBS_ols,C8,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
3,Transit,TSH,180702,34 (0.019%),124 (86 - 173),179389 (99.3%),31 (0.017%),3 (0.228%),13.2 (4.0 - 43.2),179387 (99.3%),31 (0.017%),3 (0.228%),13.2 (4.0 - 43.1)
10,Transit,IRT,246449,41 (0.017%),109 (79 - 149),242822 (98.5%),41 (0.017%),0 (0.000%),,244239 (99.1%),40 (0.016%),1 (0.045%),2.8 (0.4 - 20.1)


In [65]:
table_sep.to_csv('supp_table_separate_trans_dbs.csv', index=False)

## Supplemental table (low adjustment)

In [66]:
adjustments_low = ['Combined_low']
table_low = table_summary[
    (table_summary['adjustment'].isin(adjustments_low))
    & (table_summary['Analyte'].isin(analyte_order))
]

table_low.to_csv('supp_table_low.csv', index=False)

In [67]:
table_low

,adjustment,Analyte,n,Number changed classifications (%),Estimated UK annual incidence (95% CI),Number acceptable DBS size (8–14mm) (%),Number changed classification (8-14mm subgroup) (%),Number changed classification (< 8 mm or > 14 mm subgroup) (%),Risk ratio DBS diameter (95% CI),Number transit time < 7 days (%),Number changed classification (< 7 day transit time) (%),Number changed classification (>= 7 day transit time) (%),Risk ratio transit time (95% CI)
5,Combined_low,TSH,180702,40 (0.022%),146 (104 - 199),179389 (99.3%),38 (0.021%),2 (0.152%),7.2 (1.7 - 29.8),179387 (99.3%),39 (0.022%),1 (0.076%),3.5 (0.5 - 25.4)
12,Combined_low,IRT,246449,94 (0.038%),251 (203 - 308),242822 (98.5%),90 (0.037%),4 (0.110%),3.0 (1.1 - 8.1),244239 (99.1%),93 (0.038%),1 (0.045%),1.2 (0.2 - 8.5)
33,Combined_low,Phe,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
61,Combined_low,Met,256554,16 (0.006%),41 (24 - 67),254060 (99.0%),14 (0.006%),2 (0.080%),14.6 (3.3 - 64.0),254177 (99.1%),11 (0.004%),5 (0.210%),48.6 (16.9 - 139.8)
68,Combined_low,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
47,Combined_low,C5,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
54,Combined_low,C5DC,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
26,Combined_low,C8,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),


## Supplemental table (high adjustment)

In [68]:
adjustments_high = ['Combined_high']
table_high = table_summary[
    (table_summary['adjustment'].isin(adjustments_high))
    & (table_summary['Analyte'].isin(analyte_order))
]


table_high.to_csv('supp_table_high.csv', index=False)

In [69]:
table_high

,adjustment,Analyte,n,Number changed classifications (%),Estimated UK annual incidence (95% CI),Number acceptable DBS size (8–14mm) (%),Number changed classification (8-14mm subgroup) (%),Number changed classification (< 8 mm or > 14 mm subgroup) (%),Risk ratio DBS diameter (95% CI),Number transit time < 7 days (%),Number changed classification (< 7 day transit time) (%),Number changed classification (>= 7 day transit time) (%),Risk ratio transit time (95% CI)
6,Combined_high,TSH,180702,50 (0.028%),182 (136 - 241),179389 (99.3%),47 (0.026%),3 (0.228%),8.7 (2.7 - 28.0),179387 (99.3%),49 (0.027%),1 (0.076%),2.8 (0.4 - 20.1)
13,Combined_high,IRT,246449,149 (0.060%),398 (337 - 468),242822 (98.5%),141 (0.058%),8 (0.221%),3.8 (1.9 - 7.7),244239 (99.1%),148 (0.061%),1 (0.045%),0.7 (0.1 - 5.3)
34,Combined_high,Phe,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
62,Combined_high,Met,256554,19 (0.007%),48 (29 - 76),254060 (99.0%),17 (0.007%),2 (0.080%),12.0 (2.8 - 51.8),254177 (99.1%),13 (0.005%),6 (0.252%),49.4 (18.8 - 129.7)
69,Combined_high,Xleu,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
48,Combined_high,C5,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
55,Combined_high,C5DC,256554,1 (<0.001%),2 (0 - 14),254060 (99.0%),1 (<0.001%),0 (0.000%),,254177 (99.1%),1 (<0.001%),0 (0.000%),
27,Combined_high,C8,256554,0 (0.000%),0 (0 - 9),254060 (99.0%),0 (0.000%),0 (0.000%),,254177 (99.1%),0 (0.000%),0 (0.000%),
